In [34]:
!pip install tensorflow keras

In [35]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

In [36]:
df = pd.read_csv("/content/country_wise_latest.csv", on_bad_lines='skip')

In [37]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# **Linear Regression**

In [38]:
features = [
    'New cases',
    'New deaths',
    'New recovered',
    'Confirmed last week',
    '1 week change',
    '1 week % increase'
]

X = df[features]
y = df['Deaths']  #regression target

In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [40]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [41]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

LinearRegression()

In [42]:
y_pred_lr = lr_model.predict(X_test)

In [43]:
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression Results:")
print("MSE:", mse_lr)
print("RMSE:", rmse_lr)

Linear Regression Results:
MSE: 62079702.4538038
RMSE: 7879.06735938993


# **Neural Network**

Build Model

In [44]:
model = Sequential()

model.add(Dense(64, activation='relu', input_dim=X_train.shape[1]))
model.add(Dense(32, activation='relu'))

# IMPORTANT: Linear output for regression
model.add(Dense(1, activation='linear'))

Compile

In [45]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

In [46]:
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - loss: 299972352.0000 - mae: 4087.4036 - val_loss: 13758278.0000 - val_mae: 1128.5332
Epoch 2/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 299956160.0000 - mae: 4087.1479 - val_loss: 13756978.0000 - val_mae: 1128.3682
Epoch 3/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 299942944.0000 - mae: 4086.9377 - val_loss: 13755496.0000 - val_mae: 1128.1957
Epoch 4/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 299933984.0000 - mae: 4086.7375 - val_loss: 13754099.0000 - val_mae: 1128.0017
Epoch 5/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 299920736.0000 - mae: 4086.5007 - val_loss: 13752582.0000 - val_mae: 1127.7872
Epoch 6/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 299908064.0000 - mae: 4086.2034 - val_loss: 13750616.0000 - val_mae: 1127.5187
Epoch 7/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 299888000.0000 - mae: 4085.8467 - val_loss: 13748542.0000 - val_mae: 1127.2241
Epoch 8/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 

In [47]:
y_pred_nn = model.predict(X_test)


1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step


In [48]:
mse_nn = mean_squared_error(y_test, y_pred_nn)
rmse_nn = np.sqrt(mse_nn)
r2_nn = r2_score(y_test, y_pred_nn)

print("\nNeural Network Results:")
print("MSE:", mse_nn)
print("RMSE:", rmse_nn)
print("R2 Score:", r2_nn)


Neural Network Results:
MSE: 104622864.0
RMSE: 10228.531859460574
R2 Score: -0.10958623886108398


Compare Results

In [49]:
print("\n===== COMPARISON =====")

print("Linear Regression -> MSE:", mse_lr, " RMSE:", rmse_lr, " R2:", r2_lr)
print("Neural Network  -> MSE:", mse_nn, " RMSE:", rmse_nn, " R2:", r2_nn)


===== COMPARISON =====
Linear Regression -> MSE: 62079702.4538038  RMSE: 7879.06735938993  R2: 0.3416087402183292
Neural Network  -> MSE: 104622864.0  RMSE: 10228.531859460574  R2: -0.10958623886108398


In [52]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

# ==============================================================
# STEP 1: Load & Clean
# ==============================================================
df = pd.read_csv('/content/country_wise_latest.csv', on_bad_lines='skip')
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
print(f"Original shape: {df.shape}")

# ==============================================================
# STEP 2: FIX — Remove Data Leakage
# 'New deaths' is derived from 'Deaths' → remove it!
# ==============================================================
features = [
    'Confirmed', 'Recovered', 'Active',
    'New cases',                          # New deaths REMOVED
    'New recovered',
    'Deaths / 100 Cases',
    'Recovered / 100 Cases',
    'Deaths / 100 Recovered',
    'Confirmed last week',
    '1 week change',
    '1 week % increase'
]

# ==============================================================
# STEP 3: FIX — Log Transform target to handle outliers
# Deaths range: 0 to 148,000 → log makes it 0 to 11.9
# ==============================================================
df['Deaths_log'] = np.log1p(df['Deaths'])   # log1p = log(1+x) handles 0 values

X = df[features]
y = df['Deaths_log']   # predict log of Deaths

print(f"Deaths range (original): {df['Deaths'].min()} to {df['Deaths'].max()}")
print(f"Deaths range (log):      {y.min():.2f} to {y.max():.2f}")

# ==============================================================
# STEP 4: Train-Test Split
# ==============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==============================================================
# STEP 5: FIX — Scale both X AND y
# ==============================================================
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled  = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_scaled  = scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

# ==============================================================
# STEP 6: Linear Regression (Fixed — no leakage)
# ==============================================================
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train_scaled)
y_pred_lr_scaled = lr_model.predict(X_test_scaled)

# Convert predictions back to original Deaths scale
y_pred_lr_log = scaler_y.inverse_transform(y_pred_lr_scaled.reshape(-1, 1)).ravel()
y_pred_lr_orig = np.expm1(y_pred_lr_log)   # reverse log transform
y_test_orig    = np.expm1(y_test.values)   # actual Deaths

mse_lr  = mean_squared_error(y_test_orig, y_pred_lr_orig)
rmse_lr = np.sqrt(mse_lr)
r2_lr   = r2_score(y_test_orig, y_pred_lr_orig)

print(f"\nLinear Regression (Fixed):")
print(f"  MSE:  {mse_lr:,.0f}")
print(f"  RMSE: {rmse_lr:,.0f}")
print(f"  R2:   {r2_lr:.4f}")

# ==============================================================
# STEP 7: Neural Network (Fixed — Dropout + EarlyStopping)
# ==============================================================
model = Sequential([
    Dense(64, activation='relu', input_dim=X_train_scaled.shape[1]),
    Dropout(0.2),          # FIX: prevents overfitting
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='linear')   # linear output for regression
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

# FIX: EarlyStopping stops training when val_loss stops improving
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train_scaled, y_train_scaled,
    epochs=200,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

print(f"NN stopped at epoch: {len(history.history['loss'])}")

# Predictions — convert back to original Deaths scale
y_pred_nn_scaled = model.predict(X_test_scaled, verbose=0).ravel()
y_pred_nn_log    = scaler_y.inverse_transform(y_pred_nn_scaled.reshape(-1, 1)).ravel()
y_pred_nn_orig   = np.expm1(y_pred_nn_log)

mse_nn  = mean_squared_error(y_test_orig, y_pred_nn_orig)
rmse_nn = np.sqrt(mse_nn)
r2_nn   = r2_score(y_test_orig, y_pred_nn_orig)

print(f"\nNeural Network (Fixed):")
print(f"  MSE:  {mse_nn:,.0f}")
print(f"  RMSE: {rmse_nn:,.0f}")
print(f"  R2:   {r2_nn:.4f}")


print("\n========== FINAL COMPARISON ==========")
print(f"{'Model':<25} {'RMSE':>12} {'R2':>8}")
print("-" * 47)
print(f"{'LR  (Broken - leakage)':<25} {'~0':>12} {'1.0':>8}  ← FAKE")
print(f"{'NN  (Broken)':<25} {10213:>12,.0f} {-0.106:>8.3f}  ← BAD")
print(f"{'LR  (Fixed)':<25} {rmse_lr:>12,.0f} {r2_lr:>8.3f}")
print(f"{'NN  (Fixed)':<25} {rmse_nn:>12,.0f} {r2_nn:>8.3f}")
print("\nPlot saved!")

Original shape: (182, 15)
Deaths range (original): 0 to 148011
Deaths range (log):      0.00 to 11.91

Linear Regression (Fixed):
  MSE:  48,717,971
  RMSE: 6,980
  R2:   0.4833
NN stopped at epoch: 82

Neural Network (Fixed):
  MSE:  33,244,275
  RMSE: 5,766
  R2:   0.6474

========== FINAL COMPARISON ==========
Model                             RMSE       R2
-----------------------------------------------
LR  (Broken - leakage)              ~0      1.0  ← FAKE
NN  (Broken)                    10,213   -0.106  ← BAD
LR  (Fixed)                      6,980    0.483
NN  (Fixed)                      5,766    0.647

Plot saved!
